# 02 · LoRA fine-tuning + test predictions
Trains LoRA adapters on the train split (10% of train stories held out as dev for checkpoint choice), then answers the same 350 test pairs.
Output: `runs/<RUN_ID>/` in Drive with `adapter/`, `run_config.json`, `log_history.csv`, `dev_ids.csv`, `eval/predictions_test.csv`.

Defaults: r=16, alpha=32, dropout=0.05, all linear layers, lr 2e-4 cosine, 3 epochs, batch 8 x 2, loss on the answer only.
Runtime: L4 about 10 min, T4 about 25 min.

In [ ]:
# ---- 1. Settings ----
REPO_URL   = "https://github.com/O-sama2004/gemma-tr-temporal-inference.git"
DRIVE_ROOT = "/content/drive/MyDrive/gemma-tr-temporal-inference"
DATA_VERSION = "v1"
DATA = "/content/data"          # local copy (Drive I/O is slow)

In [ ]:
# ---- 2. Mount Drive, get the code ----
import os, subprocess, shutil
from google.colab import drive, userdata
drive.mount("/content/drive")

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if not os.path.exists("/content/repo"):
    tok = secret("GITHUB_TOKEN")             # only needed if the repo is private
    url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
os.chdir("/content/repo")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ---- 3. Pinned libraries (torch comes with Colab) ----
# If a later cell fails with an import error: Runtime > Restart session, then run cells 1, 2, 4, 5 again (skip this one).
!pip install -q -r requirements.txt

In [ ]:
# ---- 4. Hugging Face access ----
# One-time: accept the Gemma licence at https://huggingface.co/google/gemma-3-1b-it
# and add a read token as a Colab secret named HF_TOKEN (key icon on the left).
assert secret("HF_TOKEN"), "add the HF_TOKEN secret and allow this notebook to use it"
os.environ["HF_TOKEN"] = secret("HF_TOKEN")

In [ ]:
# ---- 5. Copy the frozen dataset to local disk and verify it ----
shutil.copytree(f"{DRIVE_ROOT}/data/{DATA_VERSION}", DATA, dirs_exist_ok=True)
!python src/verify_data.py {DATA}

In [ ]:
# ---- 6. GPU check ----
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), "| dtype:", "bf16" if torch.cuda.is_bf16_supported() else "fp32 (T4: slower, still correct)")

## 7. Smoke test (5 steps)
Read the printed example: the "tokens with loss" block must contain only `<start_of_turn>model`, the answer and `<end_of_turn>`. The story and the question must not appear there.

In [ ]:
!python src/train.py --data_dir {DATA} --out_dir /content/smoke_run --max_steps 5

## 8. Full training

In [ ]:
import datetime
LR, R, EPOCHS = 2e-4, 16, 3
RUN_ID = f"{datetime.date.today()}_lora_r{R}_lr{LR:g}_ep{EPOCHS}"
RUN = f"{DRIVE_ROOT}/runs/{RUN_ID}"
print(RUN)
!python src/train.py --data_dir {DATA} --out_dir "{RUN}" --lr {LR} --lora_r {R} --lora_alpha {2*R} --epochs {EPOCHS}

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv(f"{RUN}/log_history.csv")
tr, ev = h.dropna(subset=["loss"]), h.dropna(subset=["eval_loss"])
plt.plot(tr.step, tr.loss, label="train"); plt.plot(ev.step, ev.eval_loss, "o-", label="dev")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()
ev[["epoch", "eval_loss"]]

## 9. Test predictions with the trained adapter

In [ ]:
OUT = f"{RUN}/eval/predictions_test.csv"
!python src/infer.py --data_dir {DATA} --split test --system finetuned --adapter "{RUN}/adapter" --out "{OUT}"
p = pd.read_csv(OUT, keep_default_na=False)
p.sample(10, random_state=0)[["question", "answer", "prediction"]]